# Getting Started with SocialMapper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/01-getting-started.ipynb)

This notebook introduces SocialMapper - a Python library for geographic accessibility analysis. You'll learn:

- How to install SocialMapper
- Core concepts: isochrones, POIs, census data
- Running your first analysis
- Using demo mode for testing

## Installation

First, let's install SocialMapper with all optional dependencies for the full feature set.

In [ ]:
# Install SocialMapper with routing support for fast isochrones
!pip install -q socialmapper[routing]

## API Keys Setup

SocialMapper uses several APIs. For this tutorial, we'll use **demo mode** which doesn't require any keys.

For production use, you'll want to set up:
- **Census API Key**: Free from https://api.census.gov/data/key_signup.html
- **ORS API Key** (optional): Free from https://openrouteservice.org/dev/

You can set these in Colab using Secrets (key icon in the left sidebar) or as environment variables.

In [ ]:
import os

# Enable demo mode for this tutorial (no API keys needed)
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

# For production, uncomment and set your keys:
# os.environ["CENSUS_API_KEY"] = "your-census-api-key"
# os.environ["ORS_API_KEY"] = "your-ors-api-key"

print("Environment configured!")

## Core Concepts

SocialMapper provides five main functions:

| Function | Purpose |
|----------|----------|
| `create_isochrone()` | Generate travel-time polygons |
| `get_poi()` | Find points of interest |
| `get_census_blocks()` | Get census geography |
| `get_census_data()` | Retrieve demographics |
| `create_map()` | Create visualizations |

In [ ]:
# Import all main functions
from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map
)

print("SocialMapper imported successfully!")

## Your First Isochrone

An **isochrone** is a polygon showing all areas reachable within a certain travel time. Let's create one for a 15-minute drive.

In [ ]:
# Create a 15-minute driving isochrone
isochrone = create_isochrone(
    location="Seattle, WA",
    travel_time=15,
    travel_mode="drive"
)

# Explore the result
print(f"Isochrone type: {isochrone['type']}")
print(f"Travel time: {isochrone['properties']['travel_time']} minutes")
print(f"Travel mode: {isochrone['properties']['travel_mode']}")
print(f"Area covered: {isochrone['properties']['area_sq_km']:.2f} km²")

## Finding Points of Interest

Let's find healthcare facilities near a location.

In [ ]:
# Find hospitals near Seattle
healthcare = get_poi(
    location="Seattle, WA",
    categories=["healthcare"],
    limit=10
)

print(f"Found {len(healthcare)} healthcare facilities:")
for h in healthcare[:5]:
    print(f"  - {h['name']}: {h['distance_km']:.2f} km away")

## Getting Census Data

Retrieve demographic information for an area.

In [ ]:
# Get census blocks within the isochrone
blocks = get_census_blocks(polygon=isochrone)
print(f"Found {len(blocks)} census block groups")

# Get population data
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(
    location=geoids,
    variables=["population"]
)

# Calculate total population
total_pop = sum(
    data.get("population", 0) or 0
    for data in census_result.data.values()
)
print(f"Total population in area: {total_pop:,}")

## Complete Analysis Example

Let's put it all together: analyze grocery store access for an area.

In [ ]:
# Define our study area
location = "Portland, OR"

print(f"Analyzing grocery access in {location}...")
print("=" * 50)

# Step 1: Create walking isochrone (15 minutes)
walk_area = create_isochrone(
    location=location,
    travel_time=15,
    travel_mode="walk"
)
print(f"\n1. Walking area: {walk_area['properties']['area_sq_km']:.2f} km²")

# Step 2: Find grocery stores
groceries = get_poi(
    location=location,
    categories=["shopping"],
    travel_time=15,
    limit=50
)
print(f"2. Grocery stores found: {len(groceries)}")

# Step 3: Get census data
blocks = get_census_blocks(polygon=walk_area)
geoids = [b['geoid'] for b in blocks]
census = get_census_data(geoids, variables=["population", "median_income"])

# Step 4: Calculate statistics
total_pop = sum(
    d.get("population", 0) or 0
    for d in census.data.values()
)
incomes = [
    d["median_income"]
    for d in census.data.values()
    if d.get("median_income")
]

print(f"3. Population with walkable access: {total_pop:,}")
if incomes:
    print(f"4. Average median income: ${sum(incomes)/len(incomes):,.0f}")

# Summary
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Location: {location}")
print(f"Travel mode: 15-minute walk")
print(f"Grocery stores accessible: {len(groceries)}")
print(f"Population served: {total_pop:,}")
if len(groceries) >= 3:
    print("Assessment: Good grocery access")
else:
    print("Assessment: Limited grocery access - potential food desert")

## Working with Coordinates

You can also use latitude/longitude coordinates instead of place names.

In [ ]:
# Using coordinates (lat, lon)
space_needle = (47.6205, -122.3493)

isochrone = create_isochrone(
    location=space_needle,
    travel_time=10,
    travel_mode="walk"
)

print(f"10-minute walk from Space Needle:")
print(f"  Area: {isochrone['properties']['area_sq_km']:.2f} km²")

## Next Steps

You've learned the basics of SocialMapper! Continue with:

1. **[Isochrone Analysis](02-isochrone-analysis.ipynb)** - Deep dive into travel-time analysis
2. **[Points of Interest](03-points-of-interest.ipynb)** - Advanced POI queries
3. **[Census Data](04-census-data.ipynb)** - Working with demographics
4. **[Mapping](05-mapping-visualization.ipynb)** - Creating visualizations
5. **[Complete Workflow](06-complete-workflow.ipynb)** - End-to-end analysis
6. **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Real-world application

## Disable Demo Mode for Production

When you're ready to use real data, disable demo mode and set your API keys.

In [ ]:
# Disable demo mode for production use
# os.environ.pop("SOCIALMAPPER_DEMO_MODE", None)
# os.environ["CENSUS_API_KEY"] = "your-key-here"

print("For production use:")
print("1. Get a Census API key from: https://api.census.gov/data/key_signup.html")
print("2. Set it as CENSUS_API_KEY environment variable")
print("3. Remove SOCIALMAPPER_DEMO_MODE from environment")